In [1]:
!pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml

In [7]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

from google.colab import userdata
userdata.get('hf_token')

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.10.0+cu128
CUDA available: True


In [8]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
CONFIG = {
    "model_name": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",  # Verify exact ID from the linked Unsloth notebook.
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "num_train_epochs": 5,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 12000,
    "eval_size": 0.02,
    "output_dir": "/kaggle/working/qwen2b_svg_lora",
}

CONFIG

{'model_name': 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit',
 'max_seq_length': 2048,
 'lora_r': 16,
 'lora_alpha': 16,
 'learning_rate': 0.0002,
 'num_train_epochs': 5,
 'per_device_train_batch_size': 2,
 'gradient_accumulation_steps': 8,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'max_train_samples_per_source': 12000,
 'eval_size': 0.02,
 'output_dir': '/kaggle/working/qwen2b_svg_lora'}

In [9]:
# Data catalog using the resources listed in contest_docs/03_Data_Design.md.
DATASET_CATALOG = {
    "OmniSVG/MMSVG-Icon": {
        "split": "train",
        "prompt_fields": ["description", "keywords", "detail", "prompt", "text"],
        "svg_fields": ["svg", "picosvg", "completion", "target"],
    },
    "xingxm/SVGX-Core-250k": {
        "split": "train",
        "prompt_fields": ["qwen_caption", "blip_caption", "name", "img_analysis", "prompt"],
        "svg_fields": ["svg", "comfpletion", "target"],
    },
    "xingxm/SVGX-SFT-1M": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input", "query"],
        "svg_fields": ["completion", "output", "svg", "response"],
    },
    "thesantatitan/deepseek-svg-dataset": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input"],
        "svg_fields": ["completion", "output", "svg"],
    },
}

# For a first run, keep to 1-2 sources.
ACTIVE_SOURCES = [
    "xingxm/SVGX-SFT-1M",
    "OmniSVG/MMSVG-Icon",
]

In [10]:
def _pick_first_non_empty(example, keys):
    for key in keys:
        if key in example and example[key] is not None:
            val = str(example[key]).strip()
            if val:
                return val
    return ""


def to_prompt_svg(example, prompt_fields, svg_fields):
    prompt = _pick_first_non_empty(example, prompt_fields)
    svg = _pick_first_non_empty(example, svg_fields)
    if not svg.lower().startswith("<svg"):
        return {"prompt": "", "svg": ""}
    return {"prompt": prompt, "svg": svg}


def load_source_dataset(dataset_id, cfg, max_samples):
    print(f"Loading {dataset_id} ...")
    ds = load_dataset(dataset_id, split=cfg["split"])
    if max_samples and len(ds) > max_samples:
        ds = ds.shuffle(seed=SEED).select(range(max_samples))
    ds = ds.map(
        lambda ex: to_prompt_svg(ex, cfg["prompt_fields"], cfg["svg_fields"]),
        remove_columns=ds.column_names,
        desc=f"normalizing {dataset_id}",
    )
    ds = ds.filter(lambda x: bool(x["prompt"]) and bool(x["svg"]))
    print(f"{dataset_id}: {len(ds)} usable rows")
    return ds

In [11]:
datasets_ok = []
for source in ACTIVE_SOURCES:
    try:
        ds = load_source_dataset(
            source,
            DATASET_CATALOG[source],
            CONFIG["max_train_samples_per_source"],
        )
        datasets_ok.append(ds)
    except Exception as e:
        print(f"Skipping {source}: {type(e).__name__}: {e}")

if not datasets_ok:
    raise RuntimeError("No dataset loaded. Check dataset IDs, internet access, and schema fields.")

train_raw = datasets_ok[0] if len(datasets_ok) == 1 else concatenate_datasets(datasets_ok)
train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[0]

Loading xingxm/SVGX-SFT-1M ...


Generating train split: 0 examples [00:00, ? examples/s]

Skipping xingxm/SVGX-SFT-1M: DatasetGenerationCastError: An error occurred while generating the dataset

All the data files must have the same columns, but at some point there are 2 new columns ({'messages', 'images'}) and 3 missing columns ({'instruction', 'output', 'input'}).

This happened while the json dataset builder was generating data using

hf://datasets/xingxm/SVGX-SFT-1M/SVGX_SFT_vision_25k.json (at revision efe8ba02df9c6be94bfcc6d96f48f3a85da41efa)

Please either edit the data files to have matching columns, or separate them into different configurations (see docs at https://hf.co/docs/hub/datasets-manual-configuration#multiple-configurations)
Loading OmniSVG/MMSVG-Icon ...


Resolving data files:   0%|          | 0/91 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/38 [00:00<?, ?it/s]

OmniSVG/MMSVG-Icon: 12000 usable rows
Train rows: 11760
Eval rows: 240


{'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#000000" fill-opacity="1.0"  filling="0" d="M169.921875 12.5 C179.5330047607422 12.5 187.34237670898438 20.213470458984375 187.49765014648438 29.787498474121094 L187.5 30.078125 L187.5 169.921875 C187.5 179.5330047607422 179.78652954101562 187.34237670898438 170.21249389648438 187.49765014648438 L169.921875 187.5 L64.453125 187.5 C61.21699142456055 187.5 58.59375 184.8767547607422 58.59375 181.640625 C58.59375 178.43692016601562 61.16484451293945 175.8337860107422 64.35624694824219 175.78204345703125 L64.453125 175.78125 L169.921875 175.78125 C173.12557983398438 175.78125 175.7287139892578 173.2101593017578 175.78045654296875 170.01873779296875 L175.78125 169.921875 L175.78125 30.078125 C175.78125 26.874412536621094 173.2101593017578 24.271286010742188 170.01873779296875 24.219528198242188 L169.921875 24.21875 L60.546875 24.21875 C57.343162536621094 24.21875 54.74

In [12]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

<|im_start|>system
You generate compact, valid SVG markup from user requests. Return only SVG code with a single root <svg> element.<|im_end|>
<|im_start|>user
The image contains a black outline of a notebook or document icon with four horizontal lines inside a rectangular shape, set against a white background.<|im_end|>
<|im_start|>assistant
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.


In [13]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.10: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.10 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [15]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    args=training_args,
)

train_result = trainer.train()
train_result

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/11760 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/240 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,760 | Num Epochs = 5 | Total steps = 3,675
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss,Validation Loss
100,0.538670,0.527117
200,0.434465,0.450005
300,0.397947,0.408027
400,0.398659,0.386212
500,0.366926,0.371893
600,0.376040,0.363713
700,0.350278,0.353402
800,0.339868,0.346629
900,0.344102,0.342075
1000,0.326751,0.336287


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transform

TrainOutput(global_step=3675, training_loss=0.32505644091132546, metrics={'train_runtime': 10363.3085, 'train_samples_per_second': 5.674, 'train_steps_per_second': 0.355, 'total_flos': 4.786898213357752e+17, 'train_loss': 0.32505644091132546, 'epoch': 5.0})

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
CONFIG["output_dir"] = "/content/drive/MyDrive/svg_model"
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

Saved adapter + tokenizer to: /content/drive/MyDrive/svg_model


In [21]:
FastLanguageModel.for_inference(model)

SVG_REGEX = re.compile(r"<svg[\\s\\S]*?</svg>", flags=re.IGNORECASE)


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False


def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


def generate_svg(prompt, max_new_tokens=512):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
        )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    svg = extract_svg(decoded)
    if not is_valid_svg(svg):
        svg = fallback_svg(prompt)
    return svg


test_prompt = "a simple blue bird icon"
pred_svg = generate_svg(test_prompt)
print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RuntimeError: output with shape [1, 12, 1, 128] doesn't match the broadcast shape [1, 12, 43, 128]

In [22]:
# Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
TEST_PROMPTS_PATH = "/kaggle/input/svg-test-public-prompts/test_prompts.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

test_df = pd.read_csv(TEST_PROMPTS_PATH)

rows = []
invalid_count = 0
t0 = time.time()

for _, row in test_df.iterrows():
    svg = generate_svg(row["prompt"])
    if not is_valid_svg(svg):
        invalid_count += 1
        svg = fallback_svg(row["prompt"])
    rows.append({"id": row["id"], "svg": svg})

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/svg-test-public-prompts/test_prompts.csv'